# Collecting the job ads

The test in `00_api_test.ipynb` showed that the job search of the Bundesagentur
für Arbeit works. This notebook collects the ads for several data job titles and
saves them as raw data.

The collection runs in two stages:

1. **Search:** for each job title, page through the results and collect the
   summaries of the ads (title, employer, location, reference number …).
2. **Details:** for each ad, fetch the full text using its reference number.

The raw data is saved to `data/raw/`. That folder is excluded from Git, so the
data is not published on GitHub — anyone can recreate it with this notebook.

## Setup

The settings for the API come from the test notebook: the `v6` endpoint for the
search, the `v4` endpoint for the details, and the headers that made the
requests work.

The path to `data/raw/` is built from the notebook's own location, so it stays
correct no matter where the notebook is run from.

In [ ]:
import base64
import json
import time
from pathlib import Path

import pandas as pd
import requests

SEARCH_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v6/jobs"
DETAIL_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobdetails/{}"

HEADERS = {
    "X-API-Key": "jobboerse-jobsuche",
    "User-Agent": "Jobsuche/2.9.2 (de.arbeitsagentur.jobboerse; build:1077; iOS 15.1.0) Alamofire/5.4.4",
}

# The notebook lives in notebooks/, so the project root is one level up.
RAW_DIR = Path.cwd().parent / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Safety switch. True = delete the saved data and collect everything again.
# False = only read the saved data; nothing is deleted or downloaded.
FRESH_RUN = False

## Requests that can fail

Collecting full texts means one request per ad — hundreds of requests in total.
Some of them will fail now and then: a slow connection, a short server problem.
Without a plan for that, one failed request would stop the whole collection.

`get_with_retry` sends a request and, if it fails, tries again up to three
times, waiting a little longer after each attempt. If all attempts fail, it
returns `None` instead of crashing, so the collection can skip that one ad and
continue.

In [54]:
def get_with_retry(url, params=None, attempts=3, pause=1.0):
    """
    Send a GET request and retry a few times if it fails.
    Args:
        url (str): The URL to send the GET request to.
        params (dict, optional): Query parameters for the request.
        attempts (int, optional): Number of retry attempts.
        pause (float, optional): Pause duration between attempts in seconds.
    Returns:
        requests.Response or None: The response object if the request was successful, otherwise None.
    """
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if response.status_code == 200:
                return response
            print(f"  status {response.status_code}, attempt {attempt}/{attempts}")
        except requests.RequestException as error:
            print(f"  {error}, attempt {attempt}/{attempts}")
        # wait a little longer after each failed attempt
        time.sleep(pause * attempt)
    return None

## Stage 1, step 1: Searching for one job title

The search returns at most one page of results at a time. `search_jobs` asks for
page 1, then page 2, and so on, until a page comes back empty.

Two details:

- `angebotsart = 1` limits the search to regular jobs, without apprenticeships,
  internships or self-employment.
- `max_results` stops the search early. That keeps test runs small and fast.

Each ad gets an extra field `suchbegriff`, which records the search term that
found it.

In [55]:
def search_jobs(term, max_results=None, page_size=100, pause=0.5):
    """Collect the search results (summaries) for one job title. 
    
    Args:
        term (str): The job title to search for.
        max_results (int, optional): Maximum number of results to collect.
        page_size (int, optional): Number of results per page.
        pause (float, optional): Pause duration between page requests in seconds.

    Returns:
        list of dict: Collected job ads with the search term included.
    """
    ads = []
    page = 1
    while True:
        params = {"was": term, "angebotsart": 1, "size": page_size, "page": page}
        response = get_with_retry(SEARCH_URL, params=params)
        if response is None:
            print(f"  giving up on '{term}' at page {page}")
            break

        results = response.json().get("ergebnisliste", [])
        if not results:          # empty page: no more results
            break

        for ad in results:
            ad["suchbegriff"] = term
        ads.extend(results)

        if max_results and len(ads) >= max_results:
            ads = ads[:max_results]
            break

        page += 1
        time.sleep(pause)        # short pause between pages

    print(f"'{term}': {len(ads)} ads")
    return ads

A quick test run of `search_jobs` for one job title:

In [56]:
test_ads = search_jobs("Data Analyst", max_results=5)
pd.DataFrame(test_ads)[["stellenangebotsTitel", "firma", "referenznummer"]]

'Data Analyst': 5 ads


,stellenangebotsTitel,firma,referenznummer
0,Data Analyst,micronex GmbH,10001-1003770783-S
1,Data Analyst (m/w/d),CHEFS CULINAR West GmbH & Co. KG,12117-YF-51513-YF-S
2,Data Analyst (m/w/d),TechMinds GmbH,12288-4935044130-S
3,Data-Analyst (m/w/d),Dipl.-Ing. Gennadi Rouditser D.I.R.A. Arbeitsv...,10001-1003704716-S
4,Data-Analyst (m/w/d),Dipl.-Ing. Gennadi Rouditser D.I.R.A. Arbeitsv...,10001-1003485928-S


The search works: it returns exactly 5 ads, as set with `max_results`, each with
a title, an employer and a reference number.

The titles already show something that matters later. The same role appears in
three spellings: "Data Analyst", "Data Analyst (m/w/d)" and "Data-Analyst
(m/w/d)". The suffix "(m/w/d)" — männlich/weiblich/divers — is standard in
German job ads, and the hyphen varies between employers. Before titles can be
counted or compared, they will need to be **normalised**.

## Stage 1, step 2: Several job titles, without duplicates

The same ad can be found by more than one search: a "BI Analyst" position may
show up under both "Data Analyst" and "Business Intelligence". Counted twice, it
would distort every later frequency.

`collect_search_results` therefore keeps each ad only once, identified by its
reference number. If an ad was found by several terms, all of them are kept in
the list `suchbegriffe`. That information is useful in itself: it shows how
much the job titles overlap.

The result is saved to `data/raw/search_results.json`.

In [57]:
def collect_search_results(terms, max_results=None):
    """Search for several job titles and remove duplicate ads, then return the unique results.
    Args:
        terms (list of str): Job titles to search for.
        max_results (int, optional): Maximum number of results per job title.
        Returns:
            list of dict: Unique job ads with search terms included.
    """
    unique = {}
    for term in terms:
        for ad in search_jobs(term, max_results=max_results):
            refnr = ad["referenznummer"]
            if refnr in unique:
                unique[refnr]["suchbegriffe"].append(term)
            else:
                ad["suchbegriffe"] = [term]
                del ad["suchbegriff"]
                unique[refnr] = ad

    ads = list(unique.values())
    print(f"\n{len(ads)} unique ads in total")

    path = RAW_DIR / "search_results.json"
    path.write_text(json.dumps(ads, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"saved to {path}")
    return ads

## Stage 2, step 1: The full text of one ad

The details endpoint expects the reference number in Base64 encoding, as found
in the test notebook. `fetch_details` returns the complete ad, including the
description in `stellenangebotsBeschreibung`, or `None` if the request failed.

In [58]:
def fetch_details(refnr):
    """
    Fetch the full ad, including the description text.
    Args:
        refnr (str): The reference number of the job ad.

    Returns:
        dict or None: The full job ad details if the request was successful, otherwise None.
    """
    encoded = base64.b64encode(refnr.encode()).decode()
    response = get_with_retry(DETAIL_URL.format(encoded))
    return response.json() if response is not None else None

A test run:

In [59]:
detail = fetch_details(test_ads[0]["referenznummer"])
print(detail["stellenangebotsTitel"])
print(detail["stellenangebotsBeschreibung"][:500])  

Data Analyst
**Ihre Aufgaben bei uns:**

- Aufbau und Weiterentwicklung von Power BI Reports, Dashboards und KPI-Systemen
- Mitarbeit bei der Einführung und Weiterentwicklung von SAP Business One
- Erstellung und Optimierung von SQL-Abfragen, Views und datenbankbasierten Auswertungen
- Datenmodellierung sowie Datenaufbereitung mit Power Query und DAX
- Automatisierung wiederkehrender Reporting- und Datenprozesse durch Skripte
- Unterstützung von Finanzen, Vertrieb, Einkauf und Produktion bei Reporting- und D


The full description arrives, here for the first ad from the test search, a Data
Analyst position.

The text uses Markdown formatting: section headings in bold (`**Ihre Aufgaben bei uns:**`) and bullet points starting with `-`. That makes it possible to
split each ad into its sections later.

One observation that is relevant for the next step: The skills do not only appear in the
requirements section. Already the task list mentions Power BI, SAP Business One,
SQL, Power Query and DAX. So the skill extraction should **search the whole text**,
not just the section *Was wir von Ihnen erwarten*. Whether a skill is mentioned
under tasks or under requirements may itself be worth comparing.

## Stage 2, step 2: The full texts of all ads

Fetching hundreds of full texts takes a while. Two measures make this robust:

- **A pause between requests** (0.5 second), so the server is not overloaded
  and the requests are not blocked.
- **Saving as it goes.** Each ad is written to `details.jsonl` as soon as it
  arrives. If the collection stops halfway, running it again
  skips the ads that are already saved.

JSONL ("JSON Lines") is used instead of a single JSON file because new lines can 
be appended to it, without rewriting the whole file.

In [60]:
def collect_details(ads, pause=0.5):
    """Fetch the full text for every ad and save it line by line.

    Args:
        ads (list of dict): The job ads to fetch details for.
        pause (float, optional): Pause duration between requests in seconds.

    Returns:
        None
    """
    path = RAW_DIR / "details.jsonl"

    # which ads are already saved from an earlier run?
    done = set()
    if path.exists():
        with path.open(encoding="utf-8") as file:
            for line in file:
                done.add(json.loads(line)["referenznummer"])

    todo = [ad for ad in ads if ad["referenznummer"] not in done]
    print(f"{len(done)} already saved, {len(todo)} to fetch")

    failed = 0
    with path.open("a", encoding="utf-8") as file:   # "a" = append
        for i, ad in enumerate(todo, start=1):
            detail = fetch_details(ad["referenznummer"])
            if detail is None:
                failed += 1
            else:
                detail["referenznummer"] = ad["referenznummer"]
                file.write(json.dumps(detail, ensure_ascii=False) + "\n")

            if i % 50 == 0:
                print(f"  {i}/{len(todo)}")
            time.sleep(pause)

    print(f"done: {len(todo) - failed} fetched, {failed} failed")

## Test run

Before the full collection, a small run with 20 ads per job title checks that
both stages work together.

The test only runs when `FRESH_RUN` is `True`, so these cells show no output
here. The numbers below are from the test run on 7 October 2026.

In [ ]:
if FRESH_RUN:
    # start clean, so the test is not mixed with older data
    for name in ["search_results.json", "details.jsonl"]:
        (RAW_DIR / name).unlink(missing_ok=True)

    ads = collect_search_results(["Data Analyst", "Data Scientist"], max_results=20)
    collect_details(ads)

'Data Analyst': 20 ads
'Data Scientist': 20 ads

40 unique ads in total
saved to e:\Projects_DS\data-job-ad-nlp-de\data\raw\search_results.json
2123 already saved, 1 to fetch
done: 1 fetched, 0 failed


In [62]:
# Check the collected job ad details
details = pd.read_json(RAW_DIR / "details.jsonl", lines=True)
print(details.shape)
details[["stellenangebotsTitel", "firma", "stellenangebotsBeschreibung"]].head()

(2124, 42)


,stellenangebotsTitel,firma,stellenangebotsBeschreibung
0,Data Analyst,micronex GmbH,**Ihre Aufgaben bei uns:**\n\n- Aufbau und Wei...
1,Data Analyst (m/w/d),CHEFS CULINAR West GmbH & Co. KG,"Data Analyst (m/w/d) \n\nDu liebst Zahlen, Dat..."
2,Data Analyst (m/w/d),TechMinds GmbH,Über uns TechMinds ist eine Personalberatung m...
3,Data-Analyst (m/w/d),Dipl.-Ing. Gennadi Rouditser D.I.R.A. Arbeitsv...,**Ihr Profil:**\n\n- Mindestens B1 in Deutsch ...
4,Data-Analyst (m/w/d),Dipl.-Ing. Gennadi Rouditser D.I.R.A. Arbeitsv...,**Ihr Profil:**\n\n- Mindestens B1 in Deutsch ...


Both stages work together: 20 ads per term give 40 ads with full texts. The
detail records have 39 fields, considerably more than the 17 of the search
summaries.

The first rows already show three things that matter for the next steps:

- **The texts are structured very differently.** Some begin with a bold section
  heading (*Ihre Aufgaben bei uns*, *Ihr Profil*), some with an introduction of
  the company (*Über uns*), some with a free-running sentence (*Du liebst
  Zahlen …*). Splitting ads into sections will therefore not work the same way
  for every ad.
- **Ads address the reader differently.** Some use the formal *Sie*/*Ihr*, others
  the informal *du*. This says nothing about skills, but it is a small window
  into how employers present themselves.
- **There are duplicates that the reference number does not catch.** Rows 3 and
  4 have the same employer, the same title and the same beginning of the text,
  but different reference numbers. The employer is a recruitment agency
  (*Arbeitsvermittlung*), which may post the same job more than once. These
  duplicates have to be removed later by comparing the texts themselves, so they
  do not count twice in the analysis.

Two quick checks before the full run, to confirm rather than eyeball the head of data:

In [ ]:
if FRESH_RUN:
    # do all ads have a description?
    print(details["stellenangebotsBeschreibung"].isna().sum(), "ads without a description")

    # how many ads share an identical description?
    print(details["stellenangebotsBeschreibung"].duplicated().sum(), "duplicate descriptions")

0 ads without a description
33 duplicate descriptions


Every ad has a description, and **one of the 40 descriptions is an exact
duplicate** — the pair seen in rows 3 and 4. So the suspicion is confirmed with
a number rather than by eye.

`duplicated()` only finds texts that are completely identical. If an agency
changes a single word between two postings, the check misses it. In the full
dataset this will be checked again, and near-identical texts may need a more
tolerant comparison.

## Choosing the search terms

Before the full collection: which search terms are worth using? The scope of this project, I focus on the following data roles:

- Data Analyst
- Data Scientist
- Data Engineer
- Business Intelligence
- Machine Learning

English job titles dominate in data roles, but German titles exist for some of them. It is worth verifying which German titles are actually used in practice.

In [64]:
candidates = [
    "Data Analyst", "Datenanalyst",
    "Data Scientist", "Datenwissenschaftler",
    "Data Engineer", "Dateningenieur",
    "Business Intelligence", "Machine Learning",
]
for term in candidates:
    params = {"was": term, "angebotsart": 1, "size": 1, "page": 1}
    r = get_with_retry(SEARCH_URL, params=params)
    print(f"{term:25} {r.json()['maxErgebnisse']}")
    time.sleep(0.5)

Data Analyst              336
Datenanalyst              19
Data Scientist            307
Datenwissenschaftler      0
Data Engineer             883
Dateningenieur            2
Business Intelligence     554
Machine Learning          441


| Term | Ads |
|---|---|
| Data Analyst | 336 |
| Datenanalyst | 19 |
| Data Scientist | 307 |
| Datenwissenschaftler | 0 |
| Data Engineer | 883 |
| Dateningenieur | 2 |
| Business Intelligence | 554 |
| Machine Learning | 441 |

**The English titles clearly dominate.** "Datenanalyst" finds 19 ads and
"Dateningenieur" only 2 — few, but they cost almost nothing to collect, and the
duplicate removal will show whether they add ads the English titles did not
find. "Datenwissenschaftler" finds nothing and is left out.

**Some terms are probably broader than the role.** "Data Engineer" returns 883
ads, more than Data Analyst and Data Scientist together. The search seems to
match the words anywhere in an ad, so "Data Engineer", "Business Intelligence"
and "Machine Learning" may also bring in general software or IT positions. This
is checked after the collection, for example with the occupation field
`hauptberuf`.

**Note:** "Data Analyst" now returns 336 ads, against 371 in the API test. The
difference is the filter `angebotsart = 1`, which excludes apprenticeships,
internships and self-employment — and the job market changes from day to day.

**How long will it take?** Added up, the terms return around 2500 ads. After duplicate removal it will be fewer, so this is
an upper bound. Each full text needs one request plus the half-second pause —
roughly one second per ad. That gives at most about 2,500 seconds, a little over
40 minutes.

## Full collection

Before the full collection, the test files are deleted to ensure the full run starts clean.

In [65]:
terms = [
    "Data Analyst",
    "Datenanalyst",
    "Data Scientist",
    "Data Engineer",
    "Dateningenieur",
    "Business Intelligence",
    "Machine Learning",
]

if FRESH_RUN:
    # remove earlier files so the collection starts clean
    for name in ["search_results.json", "details.jsonl"]:
        (RAW_DIR / name).unlink(missing_ok=True)
    ads = collect_search_results(terms)
else:
    # load the saved search results instead of searching again
    ads = json.loads((RAW_DIR / "search_results.json").read_text(encoding="utf-8"))
    print(f"{len(ads)} ads loaded from search_results.json")

'Data Analyst': 336 ads
'Datenanalyst': 19 ads
'Data Scientist': 307 ads
'Data Engineer': 883 ads
'Dateningenieur': 2 ads
'Business Intelligence': 554 ads
'Machine Learning': 441 ads

2121 unique ads in total
saved to e:\Projects_DS\data-job-ad-nlp-de\data\raw\search_results.json


The fresh run returned exactly the counts from the term check above, so the
paging collected every result:

| Term | Ads |
|---|---|
| Data Analyst | 336 |
| Datenanalyst | 19 |
| Data Scientist | 307 |
| Data Engineer | 883 |
| Dateningenieur | 2 |
| Business Intelligence | 554 |
| Machine Learning | 441 |
| **Total found** | **2,542** |
| **Unique ads** | **2,121** |

**421 of the 2,542 results were duplicates (16.6%)** — ads found by more than
one search term. Without the duplicate removal they would have been counted
twice. At the same time, the overlap is smaller than expected: most terms find
largely different ads.

With about 2,100 unique ads, fetching the full texts takes roughly 35 minutes.

### How much do the search terms overlap?

In [66]:
search = pd.DataFrame(ads)
search["n_terms"] = search["suchbegriffe"].apply(len)

# by how many search terms was each ad found?
print(search["n_terms"].value_counts().sort_index(), "\n")

# which ads did a German term find that its English partner did not?
for german, english in [("Datenanalyst", "Data Analyst"), ("Dateningenieur", "Data Engineer")]:
    only_german = search["suchbegriffe"].apply(lambda t: german in t and english not in t)
    print(f"{german}: {only_german.sum()} ads not found by '{english}'")

n_terms
1    1760
2     305
3      52
4       4
Name: count, dtype: int64 

Datenanalyst: 10 ads not found by 'Data Analyst'
Dateningenieur: 0 ads not found by 'Data Engineer'


| Found by | Ads |
|---|---|
| 1 term | 1,760 |
| 2 terms | 305 |
| 3 terms | 52 |
| 4 terms | 4 |

**83% of the ads were found by only one search term.** The terms therefore
cover largely different parts of the job market; none of them is redundant. The
numbers also add up with the duplicate count: 305 ads counted once extra, 52
counted twice extra and 4 counted three times extra give exactly the 421
duplicates removed above (305 + 104 + 12).

**The German titles behave differently.** "Datenanalyst" found 10 of its 19 ads
that "Data Analyst" did not — more than half. So searching in both languages
does pay off for this role. "Dateningenieur" found nothing new: both of its ads
were also found by "Data Engineer". It costs nothing to keep, but it adds
nothing either.

Now we collect the full texts of the unique ads. Running the following cell might take some time; if it stops, just run this cell again.

In [67]:
# takes roughly one second per ad - if it stops, just run this cell again
if FRESH_RUN:
    collect_details(ads)

0 already saved, 2121 to fetch
  50/2121
  100/2121
  150/2121
  200/2121
  250/2121
  300/2121
  350/2121
  400/2121
  status 404, attempt 1/3
  status 404, attempt 2/3
  status 404, attempt 3/3
  450/2121
  500/2121
  550/2121
  600/2121
  650/2121
  700/2121
  750/2121
  800/2121
  850/2121
  900/2121
  950/2121
  1000/2121
  1050/2121
  1100/2121
  1150/2121
  1200/2121
  1250/2121
  1300/2121
  1350/2121
  1400/2121
  1450/2121
  1500/2121
  status 404, attempt 1/3
  status 404, attempt 2/3
  status 404, attempt 3/3
  1550/2121
  1600/2121
  1650/2121
  1700/2121
  1750/2121
  1800/2121
  1850/2121
  1900/2121
  1950/2121
  2000/2121
  2050/2121
  2100/2121
done: 2119 fetched, 2 failed


**Result of the fresh run: 2,119 of 2,121 ads fetched, 2 failed.**

Two ads returned status 404 (not found) on all three attempts and were skipped.
Most likely they were taken down between the search and the details request —
job ads expire continuously, and retrying cannot help in that case.

So `search_results.json` holds 2,121 ads and `details.jsonl` holds 2,119. The
following steps work with the 2,119 ads that have a full text.

**A note on the numbers:** during development, the notebook was once run from
top to bottom by accident, which deleted the raw data and started the collection
again. Because the job market changes within hours, each run returned slightly
different counts. The switch `FRESH_RUN` was added for that reason, and all
numbers in this section come from one single fresh run on 7 October 2026.

Check the shape and first few rows of the collected ads:

In [69]:
details = pd.read_json(RAW_DIR / "details.jsonl", lines=True)
print(details.shape)
print(details["stellenangebotsBeschreibung"].isna().sum(), "ads without a description")
print(details["stellenangebotsBeschreibung"].duplicated().sum(), "duplicate descriptions")
details.head()

(2119, 42)
0 ads without a description
33 duplicate descriptions


,stellenangebotsart,stellenangebotsTitel,stellenangebotsBeschreibung,arbeitszeitVollzeit,eintrittszeitraum,verguetungsangabe,vertragsdauer,stellenlokationen,homeofficemoeglich,homeofficetyp,...,alternativBeruf1,alternativBeruf2,artDerVerguetung,gehaltsspanneVon,gehaltsspanneBis,festgehalt,befristungInMonaten,homeofficeprozent,weitereBerufe,befristetBis
0,ARBEIT,Data Analyst,**Ihre Aufgaben bei uns:**\n\n- Aufbau und Wei...,1.0,{'von': '2026-09-28'},KEINE_ANGABEN,UNBEFRISTET,"[{'adresse': {'strasse': 'Sölterreeke', 'hausn...",1.0,NACH_VEREINBARUNG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ARBEIT,Data Analyst (m/w/d),"Data Analyst (m/w/d) \n\nDu liebst Zahlen, Dat...",1.0,{'von': '2026-09-23'},KEINE_ANGABEN,KEINE_ANGABE,"[{'adresse': {'plz': '47652', 'ort': 'Weeze', ...",1.0,NACH_VEREINBARUNG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ARBEIT,Data Analyst (m/w/d),Über uns TechMinds ist eine Personalberatung m...,1.0,{'von': '2026-10-06'},KEINE_ANGABEN,UNBEFRISTET,"[{'adresse': {'plz': '36396', 'ort': 'Steinau ...",0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ARBEIT,Data-Analyst (m/w/d),**Ihr Profil:**\n\n- Mindestens B1 in Deutsch ...,1.0,{'von': '2026-09-14'},KEINE_ANGABEN,KEINE_ANGABE,"[{'adresse': {'plz': '10785', 'ort': 'Berlin',...",1.0,NACH_VEREINBARUNG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ARBEIT,Data-Analyst (m/w/d),**Ihr Profil:**\n\n- Mindestens B1 in Deutsch ...,1.0,{'von': '2026-09-14'},KEINE_ANGABEN,KEINE_ANGABE,"[{'adresse': {'plz': '10969', 'ort': 'Berlin',...",1.0,NACH_VEREINBARUNG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**The final dataset: 2,119 ads with 42 fields, and every ad has a description.**

The detail records now have 42 fields instead of the 39 seen in the test run.
Some fields only appear in a few ads — salary ranges (`gehaltsspanneVon`,
`gehaltsspanneBis`), fixed-term duration (`befristungInMonaten`) or the share of
home office (`homeofficeprozent`). In most ads they are empty; the first rows
show `KEINE_ANGABEN` (no information) for the salary.

Three things need work before the analysis:

- **Nested fields.** `eintrittszeitraum` and `stellenlokationen` contain
  dictionaries and lists rather than single values. They have to be unpacked,
  for example into separate columns for the start date, the city and the
  federal state.
- **Yes/no fields stored as numbers.** `arbeitszeitVollzeit` and
  `homeofficemoeglich` appear as `1.0` and `0.0`. They turn into decimals because
  some ads leave them empty, and pandas uses `NaN` for that.
- **Duplicates need a closer look.** XX descriptions are exact copies of an
  earlier one. Rows 3 and 4 have the same title, employer
  and text — but different postcodes in Berlin (10785 and 10969). So these may
  not be a job posted twice, but one job advertised for two locations. For
  counting skills, such an ad should count once; for a regional analysis, both
  locations could matter. The cleaning step has to decide this deliberately.

## Summary

The data collection is complete: **2,119 job ads with full texts**, collected on
7 October 2026 from 7 search terms. The raw data is in `data/raw/` and can be
recreated with this notebook.

Next step (`02_cleaning.ipynb`): removing duplicates, unpacking the nested
fields, normalising the job titles and preparing the texts for skill
extraction.